# Project 3 — SQL Customer Analytics & Segmentation
### The Flying Enterprise Cork | MySQL · Python · Pandas

**Author:** Bhunesh Dadheech

8 production SQL queries answering real hospitality business questions. All queries use standard MySQL syntax and run directly in MySQL Workbench.


In [1]:
import pandas as pd
import sqlite3, os, warnings
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

DATA_PATH = 'data'
OUT_PATH  = 'outputs'
DB_PATH   = os.path.join(OUT_PATH, 'p3_analytics.db')
os.makedirs(OUT_PATH, exist_ok=True)

conn = sqlite3.connect(DB_PATH)
for name in ['transactions','customers','inventory','staff_schedules']:
    pd.read_csv(f'{DATA_PATH}/{name}.csv').to_sql(name, conn, if_exists='replace', index=False)
print('Database loaded — ready to query')

Database loaded — ready to query


## Q1 — Revenue & Gross Profit by Category
**Business question:** Which product category earns the most and has the best margin?

**SQL technique:** `GROUP BY` + `SUM` + subquery for revenue share %


In [2]:
q1 = pd.read_sql_query(
    'SELECT category, COUNT(*) AS transactions, '
    'ROUND(SUM(revenue),2) AS total_revenue, '
    'ROUND(SUM(gross_profit),2) AS gross_profit, '
    'ROUND(SUM(gross_profit)*100.0/SUM(revenue),1) AS gp_margin_pct, '
    'ROUND(SUM(revenue)*100.0/(SELECT SUM(revenue) FROM transactions),1) AS revenue_share_pct '
    'FROM transactions GROUP BY category ORDER BY total_revenue DESC', conn)
print('Q1 — Revenue & GP by Category:')
print(q1.to_string(index=False))
q1.to_csv(f'{OUT_PATH}/p3_q1_category_revenue.csv', index=False)

Q1 — Revenue & GP by Category:
     category  transactions  total_revenue  gross_profit  gp_margin_pct  revenue_share_pct
         Food         45806       884934.5      583193.0           65.9               43.0
 Draught Beer         45543       431268.3      316278.3           73.3               21.0
      Spirits         27122       329953.5      242457.4           73.5               16.0
         Wine         27060       311029.0      223091.0           71.7               15.1
Non-Alcoholic         18211        99738.4       81987.2           82.2                4.8


## Q2 — Customer Segmentation (RFM-style)
**Business question:** Who are our VIP, Regular, Occasional and New customers — and what are they worth?

**SQL technique:** `GROUP BY` + multiple `AVG` + `COUNT` + percentage calculation

**Why this matters:** VIP customers (top 6%) generate disproportionate revenue — retention spend should target them first.


In [3]:
q2 = pd.read_sql_query(
    'SELECT segment, COUNT(*) AS customer_count, '
    'ROUND(AVG(total_visits),1) AS avg_visits, '
    'ROUND(AVG(avg_spend_eur),2) AS avg_spend_eur, '
    'ROUND(AVG(total_spend_eur),2) AS avg_lifetime_value, '
    'ROUND(SUM(total_spend_eur),2) AS segment_total_revenue, '
    'ROUND(COUNT(*)*100.0/(SELECT COUNT(*) FROM customers),1) AS pct_of_customers '
    'FROM customers GROUP BY segment ORDER BY avg_lifetime_value DESC', conn)
print('Q2 — Customer Segmentation:')
print(q2.to_string(index=False))
q2.to_csv(f'{OUT_PATH}/p3_q2_segmentation.csv', index=False)

Q2 — Customer Segmentation:
   segment  customer_count  avg_visits  avg_spend_eur  avg_lifetime_value  segment_total_revenue  pct_of_customers
       VIP             719        84.9          50.56             4288.53             3083456.30              59.9
   Regular             296        33.7          50.52             1707.75              505494.35              24.7
Occasional             152        11.6          47.62              551.60               83843.24              12.7
       New              33         2.4          51.86              123.84                4086.84               2.8


## Q3 — Top 10 Products with RANK() Window Function
**Business question:** Which products drive the most revenue?

**SQL technique:** `RANK() OVER (ORDER BY ...)` — a window function that assigns a rank without collapsing rows.

**Interview note:** Window functions are one of the most commonly tested SQL topics. `RANK()` vs `DENSE_RANK()` vs `ROW_NUMBER()` is a classic question.


In [4]:
q3 = pd.read_sql_query(
    'SELECT product, category, SUM(quantity) AS units_sold, '
    'ROUND(SUM(revenue),2) AS total_revenue, '
    'ROUND(SUM(gross_profit)*100.0/SUM(revenue),1) AS gp_margin_pct, '
    'RANK() OVER (ORDER BY SUM(revenue) DESC) AS revenue_rank '
    'FROM transactions GROUP BY product, category '
    'ORDER BY total_revenue DESC LIMIT 10', conn)
print('Q3 — Top 10 Products by Revenue:')
print(q3.to_string(index=False))
q3.to_csv(f'{OUT_PATH}/p3_q3_top_products.csv', index=False)

Q3 — Top 10 Products by Revenue:
        product category  units_sold  total_revenue  gp_margin_pct  revenue_rank
   Fish & Chips     Food       13469       208769.5           64.5             1
 Burger & Chips     Food       13823       200433.5           65.5             2
  Club Sandwich     Food       13720       178360.0           65.4             3
  Chicken Wings     Food       13776       165312.0           66.7             4
   Loaded Fries     Food       13901       132059.5           68.4             5
    Gin & Tonic  Spirits       13229       119061.0           72.2             6
 Prosecco Glass     Wine       13628       109024.0           68.8             7
Jameson & Mixer  Spirits       13605       108840.0           72.5             8
  Vodka & Mixer  Spirits       13607       102052.5           76.0             9
 House Red Wine     Wine       13520       101400.0           73.3            10


## Q4 — Event Day Impact
**Business question:** How much more do we earn on event days vs normal days?

**SQL technique:** `CASE WHEN` for conditional aggregation, `COUNT(DISTINCT date)` for accurate day counts.

**Why NULLIF?** `NULLIF(x, 0)` returns NULL instead of 0, preventing division-by-zero errors cleanly.


In [5]:
q5 = pd.read_sql_query(
    'SELECT CASE WHEN is_event_day=1 THEN "Event Day" ELSE "Normal Day" END AS day_type, '
    'COUNT(DISTINCT date) AS trading_days, COUNT(*) AS total_transactions, '
    'ROUND(SUM(revenue),2) AS total_revenue, '
    'ROUND(AVG(revenue),2) AS avg_per_transaction, '
    'ROUND(SUM(revenue)/NULLIF(COUNT(DISTINCT date),0),2) AS avg_daily_revenue '
    'FROM transactions GROUP BY is_event_day', conn)
print('Q4 — Event Day Impact:')
print(q5.to_string(index=False))
q5.to_csv(f'{OUT_PATH}/p3_q4_event_impact.csv', index=False)

Q4 — Event Day Impact:
  day_type  trading_days  total_transactions  total_revenue  avg_per_transaction  avg_daily_revenue
Normal Day           720              157242      1975434.8                12.56            2743.66
 Event Day            11                6500        81488.9                12.54            7408.08


## Q5 — Staff Performance (Revenue per Hour)
**Business question:** Which staff member generates the most revenue per hour worked?

**SQL technique:** `LEFT JOIN` with a subquery, `NULLIF` for safe division.


In [6]:
q6 = pd.read_sql_query(
    'SELECT t.staff_name, COUNT(*) AS transactions_served, '
    'ROUND(SUM(t.revenue),2) AS revenue_generated, '
    'ROUND(AVG(t.revenue),2) AS avg_transaction_value, '
    's.total_hours, '
    'ROUND(SUM(t.revenue)/NULLIF(s.total_hours,0),2) AS revenue_per_hour '
    'FROM transactions t '
    'LEFT JOIN (SELECT staff_name, SUM(hours_worked) AS total_hours FROM staff_schedules GROUP BY staff_name) s '
    'ON t.staff_name = s.staff_name '
    'GROUP BY t.staff_name, s.total_hours ORDER BY revenue_generated DESC LIMIT 10', conn)
print('Q5 — Staff Revenue Performance:')
print(q6.to_string(index=False))
q6.to_csv(f'{OUT_PATH}/p3_q5_staff_performance.csv', index=False)

Q5 — Staff Revenue Performance:
      staff_name  transactions_served  revenue_generated  avg_transaction_value  total_hours  revenue_per_hour
     Niamh Walsh                16548           209775.9                  12.68         2103             99.75
    Declan Byrne                16413           206402.4                  12.58         2005            102.94
   Liam McCarthy                16344           206287.1                  12.62         2192             94.11
 Seamus Connolly                16314           206167.8                  12.64         2089             98.69
 Cian O'Sullivan                16397           205501.2                  12.53         2146             95.76
    Aoife Murphy                16405           205040.1                  12.50         1968            104.19
      Emma Burke                16243           204739.4                  12.60         2025            101.11
    Roisin Doyle                16441           204537.4                  12.44 

## Q6 — Monthly YoY Comparison
**Business question:** How does each month in 2024 compare to the same month in 2023?

**SQL technique:** Conditional `SUM(CASE WHEN year=2024 THEN revenue ELSE 0 END)` — a pivot pattern used constantly in reporting. More flexible than separate queries per year.


In [7]:
q8 = pd.read_sql_query(
    'SELECT month_name, month, '
    'ROUND(SUM(CASE WHEN year=2023 THEN revenue ELSE 0 END),2) AS rev_2023, '
    'ROUND(SUM(CASE WHEN year=2024 THEN revenue ELSE 0 END),2) AS rev_2024, '
    'ROUND((SUM(CASE WHEN year=2024 THEN revenue ELSE 0 END) - '
    'SUM(CASE WHEN year=2023 THEN revenue ELSE 0 END))*100.0/ '
    'NULLIF(SUM(CASE WHEN year=2023 THEN revenue ELSE 0 END),0),1) AS yoy_growth_pct '
    'FROM transactions GROUP BY month_name, month ORDER BY month', conn)
print('Q6 — YoY Monthly Comparison:')
print(q8.to_string(index=False))
q8.to_csv(f'{OUT_PATH}/p3_q6_yoy_comparison.csv', index=False)
conn.close()

Q6 — YoY Monthly Comparison:
month_name  month  rev_2023  rev_2024  yoy_growth_pct
   January      1   60290.8   62290.9             3.3
  February      2   57096.0   61432.6             7.6
     March      3   86919.0   89938.1             3.5
     April      4   81897.0   78740.8            -3.9
       May      5   79552.4   82080.9             3.2
      June      6  115352.6  122077.4             5.8
      July      7  121008.3  108580.4           -10.3
    August      8  110817.0  116633.7             5.2
 September      9   82114.9   77889.3            -5.1
   October     10   86286.1   84598.3            -2.0
  November     11   79471.1   80712.6             1.6
  December     12   68531.3   62612.2            -8.6


## Visualisation — 4-Panel Analytics Chart


In [8]:
DARK='#0f1117'; MID='#1a1d2e'; GREEN='#00d4aa'; GOLD='#f5a623'
BLUE='#4a9eff'; ACCENT='#e94560'; WHITE='#f0f0f0'; MUTED='#8888aa'
fig, axes = plt.subplots(2, 2, figsize=(16, 11), facecolor=DARK)
fig.suptitle('FLYING ENTERPRISE CORK — SQL CUSTOMER ANALYTICS', fontsize=15, color=WHITE, fontweight='bold')

ax = axes[0][0]; ax.set_facecolor(MID)
segs = q2.copy(); segs['avg_lifetime_value'] = segs['avg_lifetime_value'].astype(float)
colors = [ACCENT,GOLD,GREEN,BLUE]
ax.barh(segs['segment'], segs['avg_lifetime_value'], color=colors, height=0.6)
ax.set_title('Customer Lifetime Value by Segment', color=WHITE, fontsize=11)
ax.tick_params(colors=MUTED)
for sp in ax.spines.values(): sp.set_color('#333355')
ax.grid(axis='x', color='#333355', alpha=0.4)

ax2 = axes[0][1]; ax2.set_facecolor(MID)
segs['customer_count'] = segs['customer_count'].astype(float)
ax2.pie(segs['customer_count'], labels=segs['segment'], autopct='%1.1f%%', colors=colors,
        startangle=90, textprops={'fontsize':9,'color':WHITE},
        wedgeprops={'linewidth':1.5,'edgecolor':DARK})
ax2.set_title('Customer Base Distribution', color=WHITE, fontsize=11)

ax3 = axes[1][0]; ax3.set_facecolor(MID)
top8 = q3.head(8).copy(); top8['total_revenue'] = top8['total_revenue'].astype(float)
bclr = [GOLD if c=='Food' else GREEN if c=='Draught Beer' else BLUE for c in top8['category']]
ax3.barh(range(len(top8)), top8['total_revenue']/1000, color=bclr, height=0.65)
ax3.set_yticks(range(len(top8)))
ax3.set_yticklabels(top8['product'], fontsize=8, color=WHITE)
ax3.set_title('Top 8 Products by Revenue', color=WHITE, fontsize=11)
ax3.tick_params(colors=MUTED)
for sp in ax3.spines.values(): sp.set_color('#333355')

ax4 = axes[1][1]; ax4.set_facecolor(MID)
yoy = q8.copy(); yoy['rev_2023'] = yoy['rev_2023'].astype(float); yoy['rev_2024'] = yoy['rev_2024'].astype(float)
x = range(len(yoy))
ax4.bar([i-0.18 for i in x], yoy['rev_2023']/1000, width=0.35, color=BLUE, label='2023', alpha=0.85)
ax4.bar([i+0.18 for i in x], yoy['rev_2024']/1000, width=0.35, color=GOLD, label='2024', alpha=0.85)
ax4.set_xticks(list(x))
ax4.set_xticklabels([m[:3] for m in yoy['month_name']], rotation=45, fontsize=8, color=MUTED)
ax4.set_title('Monthly Revenue: 2023 vs 2024', color=WHITE, fontsize=11)
ax4.legend(fontsize=9, facecolor=MID, edgecolor='#333355', labelcolor=WHITE)
ax4.tick_params(colors=MUTED)
for sp in ax4.spines.values(): sp.set_color('#333355')

plt.tight_layout(rect=[0,0,1,0.96])
plt.savefig(f'{OUT_PATH}/project3_sql_analytics.png', dpi=150, bbox_inches='tight', facecolor=DARK)
plt.close()
print('Chart saved to outputs/project3_sql_analytics.png')

Chart saved to outputs/project3_sql_analytics.png


## Interview Talking Points

- **Window functions vs GROUP BY:** `RANK() OVER()` assigns a rank per row without collapsing — `GROUP BY` collapses rows into one per group. If an interviewer asks you to rank products *within* a category, that's `PARTITION BY category ORDER BY SUM(revenue) DESC`.
- **NULLIF for safe division:** `SUM(revenue) / NULLIF(days, 0)` returns NULL instead of a divide-by-zero error. Always use it when the denominator could be 0.
- **Conditional aggregation:** `SUM(CASE WHEN year=2024 THEN revenue ELSE 0 END)` is a pivot — turning rows into columns. Interviewers love this pattern.
- **Query optimisation (from Celebal):** Mention indexing on `date`, `category`, and `staff_name` — the most frequently filtered/grouped columns. This is what delivered the 30% query speed improvement.
